In [15]:
import os
import numpy as np
import pandas as pd
import openpyxl
from pathlib import Path
import rasterio

In [16]:
df = pd.read_excel("SF3 - Features_dataset.xlsx",index_col=0)
df = df.rename(columns={"Feature_Diameter_km":"Diameter (km)"})
df2 = df.copy()

In [17]:
filename = Path("/mnt/export/lee/1-Projects/deepcaldera/source_data/gebco/gebco_2023_clipped_to_seamounts_proj.tif")
src = rasterio.open(filename)
src_data= src.read(1)

In [19]:
%run ../scripts/caldera.py
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D



In [20]:
def sort_key(idx):
    prefix, num = idx.rsplit('-', 1)
    is_scd = prefix != 'SCD'  # False (0) sorts before True (1)
    return (is_scd, prefix, int(num))

df2 = df2.sort_index(key=lambda idx: pd.Index([sort_key(i) for i in idx]))
#
#scd_first = sorted(list(df2.index[df2.index.str.contains("SCD")]))+ sorted(list(df2.index[~df2.index.str.contains("SCD")]))
#df2 = df2.loc[scd_first]

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
print(os.getcwd())
#%run scripts/caldera.py
#LEGENDS ON ALL PAGES
# Create the PdfPages object to which we will save the pages:
# The with statement makes sure that the PdfPages object is closed properly at
# the end of the block, even if an Exception occurs.
pcount=0 
nrow=3
maxpagecount=2 #only make 2 pages for the demo

#print(df.columns
df = df2.copy()#[:9]#
#df = df[scd_first]#df.sort_index()
def page_legend(fig,colors):
    from matplotlib.lines import Line2D

    # colors you already have
    cs_colors = [colors[0], colors[1], colors[2], colors[3]]  # 4 directions
    edge_color_feature = colors[4]
    edge_color_caldera = colors[5]

    # Cross sections legend (short labels)
    cs_handles = [
        Line2D([0],[0], color=cs_colors[0], lw=2, label="S–N"),
        Line2D([0],[0], color=cs_colors[1], lw=2, label="SW–NE"),
        Line2D([0],[0], color=cs_colors[2], lw=2, label="W–E"),
        Line2D([0],[0], color=cs_colors[3], lw=2, label="NW–SE"),
    ]
    leg1 = fig.legend(cs_handles, [h.get_label() for h in cs_handles],
                    title="Cross sections", loc="upper left",
                    bbox_to_anchor=(0.02, 0.98), ncol=2,
                    frameon=False, fontsize=9, title_fontproperties={'weight': 'bold','size':10})

    # Edge stats legend (mean vs range encoded by linestyle)
    edge_handles = [
        Line2D([0],[0], color=edge_color_feature, lw=2, ls='-',  label="Feature mean"),
        Line2D([0],[0], color=edge_color_feature, lw=2, ls='--', label="Feature range"),
        Line2D([0],[0], color=edge_color_caldera, lw=2, ls='-',  label="Caldera mean"),
        Line2D([0],[0], color=edge_color_caldera, lw=2, ls='--', label="Caldera range"),
    ]
    leg2 = fig.legend(edge_handles, [h.get_label() for h in edge_handles],
                    title="Depth markers", loc="upper right",
                    bbox_to_anchor=(0.98, 0.98), ncol=2,
                    frameon=False, fontsize=9, title_fontproperties={'weight': 'bold','size':10})


    title = "Feature Types"

    fig.text(
        0.45, 0.975, title,
        ha="center", va="top",
        fontsize=10,fontweight='bold'
    )
    text = "\nSCD: Caldera (full, eroded, or small) \nA:Atoll \nE:Eroded \nU:Unidentified"

    fig.text(
        0.45, 0.975, text,
        ha="center", va="top",
        fontsize=10,
        #bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="white")
    )

    # keep both
    fig.add_artist(leg1)
def old_page_legend(fig,colors):
    labels = ["cross section S-N",
                        "cross section SW-NE",
                        "cross section W-E",
                        "cross section NW-SE",
                        "mean depth at feature edge",
                        "depth range at feature edge",
                        "mean depth at caldera edge",
                        "depth range at caldera edge"]
    linestyles=["-","-","-","-","-","--","-","--"]
    colors = [colors[0],colors[1],colors[2],colors[3],colors[4],colors[4],colors[5],colors[5]]
    lines = [Line2D([0],[0],color=colors[i], linestyle=linestyles[i], label=labels[i]) for i in range(len(labels))]
    
    # Collect legend handles and labels from one subplot
    #handles, labels = axs[0, 0].get_legend_handles_labels()
    
    # Add custom lines to the handles
    handles = lines
    labels = labels
    
    # Add a figure-wide legend at the top
    #print(handles
    legend = fig.legend(handles, labels, loc="upper center", ncol=2,
    bbox_to_anchor=(0.5, 0.995),  # higher
    fontsize=9, frameon=False)            
    #plt.tight_layout(rect=[0, 0, 1.0, 0.88])  # Leave space for the top legend

with PdfPages('S2.pdf') as pdf:
    while (pcount*nrow)<len(df) and pcount < maxpagecount:
        caption=False
        caldera=False
        print(pcount, len(df)/nrow)
        my_nrow = min(len(df)-(pcount*nrow),nrow)
        fig, axs = plt.subplots(nrow,2,figsize=(2*4,nrow*3.7))
        unused = axs.copy()
        text=[]
        for i,a in enumerate(axs):
            if pcount*nrow+i >= len(df):
                a[0].set_visible(False)
                a[1].set_visible(False)
                #a[0].remove() 
                #a[1].remove()
                continue
            row = df.iloc[pcount*nrow+i]
            if (row["Caldera_Diameter_km"]=="<Null>"):
                plot_features(row, src_data, src,axs=a)
            else:
                row2 = pd.Series({"Long":row["Long"], "Lat":row["Lat"],"Diameter (km)":row["Caldera_Diameter_km"]})
                plot_features(row, src_data,src, dim=256,row2=row2,axs=a, second=False)
                caldera=True
            #a[0].text(1.25,1.0,row.name, weight='bold', transform=a[0].transAxes)
            # Get the position of the left subplot in this row
            a[0].set_aspect('equal')#, adjustable='datalim')
            a[1].set_aspect('auto')
            bbox = a[0].get_position()
            #print(bbox.x0, bbox.width, bbox.y0, bbox.height)
            x_text = bbox.x1 + 0.2*bbox.width   # 0.01 figure-units left of the left subplot
            y_text = bbox.y0 + bbox.height*0.9
            text.append(str(row.name))
            #a[0].set_aspect('equal')
            
        
        if not caption:
            caption = True
            # Create custom lines
            prop_cycler = plt.rcParams['axes.prop_cycle']
            # Extract just the colors
            colors = [d['color'] for d in prop_cycler]
            #old_page_legend(fig,colors)
            page_legend(fig,colors)
        fig.subplots_adjust(left=0.05, right=0.98, bottom=0.1, top=0.88,
                    wspace=0.25, hspace=0.50)
        
        for i,a in enumerate(axs):
            if len(text) <= i:
                continue
            bbox = a[0].get_position()
            x_text = 0.48#bbox.x1 + 0.25*bbox.width   # 0.01 figure-units left of the left subplot
            y_text = bbox.y1 + bbox.height*0.
            fig.text(x_text, y_text, text[i],
                ha="center", va="center", weight="bold", fontsize=11)
        from matplotlib.patches import Rectangle

        pdf.savefig()
        plt.close()
        pcount+=1


/mnt/export/lee/1-Projects/dctest/deepcaldera_project/Notebooks
0 171.33333333333334
1 171.33333333333334
